Время предсказания только макроэкономических показателей 40 минут

Время предсказания с макроэкономическими и аграрными показателями 1 час 45 минут


H0: Агропромышленные показатели влияют на качество модели

H1: Агропромышленные показатели не влияют на качество модели

Уровень значимости: 5%

Статистика: Тесты для связанных выборок. t-тест Стюдента для зависимых выборок / критерий знаков Вилкоксона

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pylab as plt
from scipy.stats import wilcoxon

In [ ]:
df1 = pd.read_csv('preds_agro-2021-06-02_2023-06-02.csv', index_col=0)
df1['name'] = '1'
df2 = pd.read_csv('preds_agro-2021-09-02_2023-09-02.csv', index_col=0)
df2['name'] = '2'
df3 = pd.read_csv('preds_agro-2021-09-02_2023-12-02.csv', index_col=0)
df3['name'] = '3'
df4 = pd.read_csv('preds_agro-2022-12-02_2024-03-02.csv', index_col=0)
df4['name'] = '4'
df5 = pd.read_csv('preds_agro-2023-03-02_2024-06-02.csv', index_col=0)
df5['name'] = '5'
df6 = pd.read_csv('preds_agro-2023-06-02_2024-09-02.csv', index_col=0)
df6['name'] = '6'
df7 = pd.read_csv('preds_agro-2023-09-02_2024-12-02.csv', index_col=0)
df7['name'] = '7'
df8 = pd.read_csv('preds_agro-2023-12-02_2025-03-02.csv', index_col=0)
df8['name'] = '8'
df = pd.concat((df1, df2, df3, df4, df5, df6, df7, df8), axis=0).reset_index(drop=True)

In [ ]:
df_no_agro1 = pd.read_csv('preds_no_agro-2021-06-02_2023-06-02.csv', index_col=0)
df_no_agro1['name'] = '1'
df_no_agro2 = pd.read_csv('preds_no_agro-2021-09-02_2023-09-02.csv', index_col=0)
df_no_agro2['name'] = '2'
df_no_agro3 = pd.read_csv('preds_no_agro-2021-09-02_2023-12-02.csv', index_col=0)
df_no_agro3['name'] = '3'
df_no_agro4 = pd.read_csv('preds_no_agro-2022-12-02_2024-03-02.csv', index_col=0)
df_no_agro4['name'] = '4'
df_no_agro5 = pd.read_csv('preds_no_agro-2023-03-02_2024-06-02.csv', index_col=0)
df_no_agro5['name'] = '5'
df_no_agro6 = pd.read_csv('preds_no_agro-2023-06-02_2024-09-02.csv', index_col=0)
df_no_agro6['name'] = '6'
df_no_agro7 = pd.read_csv('preds_no_agro-2023-09-02_2024-12-02.csv', index_col=0)
df_no_agro7['name'] = '7'
df_no_agro8 = pd.read_csv('preds_no_agro-2023-12-02_2025-03-02.csv', index_col=0)
df_no_agro8['name'] = '8'
df_no_agro = pd.concat((df_no_agro1, df_no_agro2, df_no_agro3, df_no_agro4, df_no_agro5, df_no_agro6, df_no_agro7, df_no_agro8), axis=0).reset_index(drop=True)

In [ ]:
df = df.rename(columns={'price':'price_agro'})

In [ ]:
df = pd.merge(df, df_no_agro[['name','date', 'price', 'product', 'product_type', 'federal_okrug']], on=['name','date', 'product', 'product_type', 'federal_okrug'])
df

In [ ]:
df = df.drop('name', axis=1)

In [ ]:
df_true = pd.read_csv('true_data.csv', index_col=0)
df_true['product'], df_true['product_type'], df_true['federal_okrug'] = zip(*df_true['ID'].apply(lambda x: x.split('|')))
df_true = df_true.rename(columns={'ds': 'date'})

In [ ]:
df = pd.merge(df, df_true[['date', 'y', 'product', 'product_type', 'federal_okrug']], on=['date', 'product', 'product_type', 'federal_okrug'], how='left')
df

In [ ]:
df = df[~df['y'].isna()]

In [ ]:
df['mape'] = np.abs((df['y'] - df['price']) / df['y']) * 100
df['mape_agro'] = np.abs((df['y'] - df['price_agro']) / df['y']) * 100

In [ ]:
plt.figure(figsize=(10, 6))
plt.hist(df['mape'], label='без аграрных показателей', bins=100, alpha=0.5)
plt.hist(df['mape_agro'], label='с аграрными показателями', bins=100, alpha=0.5)
plt.xlim(0, 100)
plt.legend()
plt.show()

In [ ]:
print(f"среднее: {round(np.mean(df['mape']), 2)} медиана: {round(np.median(df['mape']), 2)} - выборка без аграрных показателей")
print(f"среднее: {round(np.mean(df['mape_agro']), 2)} медиана: {round(np.median(df['mape_agro']), 2)} - выборка с аграрными показателями")
pvalue = wilcoxon(df['mape'], df['mape_agro']).pvalue
print(pvalue, 'Выборки из разных распределений -', pvalue < 0.05)

In [ ]:
df['ID'] = df['product'] + '|' + df['product_type'] + '|' + df['federal_okrug']

In [ ]:
for id in df['ID'].unique():
    plt.figure(figsize=(16, 6))
    df_ = df[df['ID'] == id]
    plt.plot(df_['date'], df_['price'], label='без аграрных показателей')
    plt.plot(df_['date'], df_['price_agro'], label='с аграрными показателями')
    plt.plot(df_['date'], df_['y'], label='истинные данные')
    plt.title(id)
    plt.legend()
    plt.show()